In [3]:
# ============================================
# MLflow Lab2 - End-to-End ML Pipeline
# Dataset: Wine Classification
# ============================================

import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

# --- Step 1: Load Data ---
print("STEP 1: Load Data")
df = pd.read_csv("data/wine_classification.csv")
print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")

# --- Step 2: Data Splitting ---
print("\nSTEP 2: Data Splitting")
from sklearn.model_selection import train_test_split

X = df.drop(["target"], axis=1)
y = df["target"]
X_train, X_rem, y_train, y_rem = train_test_split(X, y, train_size=0.6, random_state=123)
X_val, X_test, y_val, y_test = train_test_split(X_rem, y_rem, test_size=0.5, random_state=123)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# --- Step 3: Baseline Model ---
print("\nSTEP 3: Baseline Model - Random Forest")
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from mlflow.models.signature import infer_signature
from mlflow.utils.environment import _mlflow_conda_env
import cloudpickle

class SklearnModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model):
        self.model = model
    def predict(self, context, model_input):
        return self.model.predict(model_input)

with mlflow.start_run(run_name="untuned_random_forest"):
    n_estimators = 10
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=123)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    wrappedModel = SklearnModelWrapper(model)
    signature = infer_signature(X_train, wrappedModel.predict(None, X_train))
    conda_env = _mlflow_conda_env(
        additional_conda_deps=None,
        additional_pip_deps=[f"cloudpickle=={cloudpickle.__version__}", f"scikit-learn=={sklearn.__version__}"],
        additional_conda_channels=None,
    )
    mlflow.pyfunc.log_model("random_forest_model", python_model=wrappedModel, conda_env=conda_env, signature=signature)
    print(f"Baseline Accuracy: {acc:.4f}, F1: {f1:.4f}")

# Feature Importance
feat_imp = pd.DataFrame(model.feature_importances_, index=X_train.columns.tolist(), columns=["importance"])
print("Top 5 Features:")
print(feat_imp.sort_values("importance", ascending=False).head())

# --- Step 4: Register Model ---
print("\nSTEP 4: Register Model")
run_id = mlflow.search_runs(filter_string='tags.mlflow.runName = "untuned_random_forest"').iloc[0].run_id
model_name = "wine_classifier"
model_version = mlflow.register_model(f"runs:/{run_id}/random_forest_model", model_name)
time.sleep(10)
print(f"Registered '{model_name}' version {model_version.version}")

# --- Step 5: Promote to Production ---
print("\nSTEP 5: Promote to Production")
from mlflow.tracking import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(name=model_name, version=model_version.version, stage="Production")
prod_model = mlflow.pyfunc.load_model(f"models:/{model_name}/production")
print(f"Production Model Accuracy: {accuracy_score(y_test, prod_model.predict(X_test)):.4f}")

# --- Step 6: Hyperparameter Tuning ---
print("\nSTEP 6: XGBoost + Hyperopt Tuning (10 trials)")
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from hyperopt.pyll import scope
import mlflow.xgboost
import xgboost as xgb

search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 15, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "objective": "multi:softmax",
    "num_class": 3,
    "seed": 123,
}

def train_model(params):
    mlflow.xgboost.autolog()
    with mlflow.start_run(nested=True):
        train = xgb.DMatrix(data=X_train, label=y_train)
        validation = xgb.DMatrix(data=X_val, label=y_val)
        booster = xgb.train(params=params, dtrain=train, num_boost_round=50,
                            evals=[(validation, "validation")], early_stopping_rounds=10, verbose_eval=False)
        val_preds = booster.predict(validation)
        val_acc = accuracy_score(y_val, val_preds)
        mlflow.log_metric("accuracy", val_acc)
        signature = infer_signature(X_train, booster.predict(train))
        mlflow.xgboost.log_model(booster, "model", signature=signature)
        return {"status": STATUS_OK, "loss": -1 * val_acc}

trials = Trials()
with mlflow.start_run(run_name="xgboost_models"):
    best_params = fmin(fn=train_model, space=search_space, algo=tpe.suggest, max_evals=10, trials=trials)
print(f"Best params: {best_params}")

# --- Step 7: Promote Best Model ---
print("\nSTEP 7: Promote Best XGBoost Model")
best_run = mlflow.search_runs(order_by=["metrics.accuracy DESC"]).iloc[0]
print(f"Best Accuracy: {best_run['metrics.accuracy']:.4f}")
new_model_version = mlflow.register_model(f"runs:/{best_run.run_id}/model", model_name)
time.sleep(10)
client.transition_model_version_stage(name=model_name, version=model_version.version, stage="Archived")
client.transition_model_version_stage(name=model_name, version=new_model_version.version, stage="Production")
print(f"Model version {new_model_version.version} promoted to Production!")

# --- Step 8: Final Evaluation ---
print("\nSTEP 8: Final Evaluation")
final_model = mlflow.pyfunc.load_model(f"models:/{model_name}/production")
print(f"Final Accuracy: {accuracy_score(y_test, final_model.predict(X_test)):.4f}")

# --- Step 9: Serve ---
print("\nSTEP 9: To serve, run in terminal:")
print(f"mlflow models serve --env-manager=local -m models:/{model_name}/production -h 0.0.0.0 -p 5001")
print("\nDONE!")

c:\Users\nishi\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/03/31 23:34:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


STEP 1: Load Data
Dataset shape: (178, 14)
Target distribution:
target
1    71
0    59
2    48
Name: count, dtype: int64

STEP 2: Data Splitting
Train: (106, 13), Val: (36, 13), Test: (36, 13)

STEP 3: Baseline Model - Random Forest


2026/03/31 23:34:26 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Registered model 'wine_classifier' already exists. Creating a new version of this model...
2026/03/31 23:34:26 WARNING mlflow.tracking._model_registry.fluent: Run with id 8c0399ca26e04f9e9e6df2f7baee7745 has no artifacts at artifact path 'random_forest_model', registering model based on models:/m-5a1eb13e929d45b0b4909f985e6eecfe instead


Baseline Accuracy: 0.9167, F1: 0.9160
Top 5 Features:
                 importance
proline            0.214986
color_intensity    0.190511
flavanoids         0.164677
alcohol            0.129436
hue                0.097895

STEP 4: Register Model


Created version '2' of model 'wine_classifier'.


...

100%|██████████| 10/10 [04:32<00:00, 27.24s/trial, best loss: -0.9722222222222222]

Registered model 'wine_classifier' already exists. Creating a new version of this model...
2026/03/31 23:39:11 WARNING mlflow.tracking._model_registry.fluent: Run with id 449928c8ef034158a8f064994940fd7c has no artifacts at artifact path 'model', registering model based on models:/m-5ab9e6d8c79b47aabd29d3f6c95b794c instead



Best params: {'learning_rate': np.float64(0.1635429911537468), 'max_depth': np.float64(12.0), 'reg_alpha': np.float64(0.29258268443597263), 'reg_lambda': np.float64(0.3668755360947658)}

STEP 7: Promote Best XGBoost Model
Best Accuracy: 0.9722


Created version '3' of model 'wine_classifier'.


Model version 3 promoted to Production!

STEP 8: Final Evaluation
Final Accuracy: 1.0000

STEP 9: To serve, run in terminal:
mlflow models serve --env-manager=local -m models:/wine_classifier/production -h 0.0.0.0 -p 5001

DONE!


In [2]:
pip install hyperopt xgboost

  Using cached future-1.0.0-py3-none-any.whl.metadata (4.0 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
    --------------------------------------- 0.0/1.6 MB 435.7 kB/s eta 0:00:04
   - -------------------------------------- 0.1/1.6 MB 544.7 kB/s eta 0:00:03
   ----- ---------------------------------- 0.2/1.6 MB 1.4 MB/s eta 0:00:01
   -------------------- ------------------- 0.8/1.6 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------  1.6/1.6 MB 7.2 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 6.3 MB/s eta 0:00:00
Using cached future-1.0.0-py3-none-any.whl (491 kB)
   ---------------------------------------- 0.0/203.0 kB ? eta -:--:--
   ---------------------------------------- 203.0/203.0 kB 6.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
